In [1]:
!pip install xgboost optuna --quiet

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import warnings, os, pickle
from pathlib import Path
import yfinance as yf

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")

In [3]:
np.random.seed(20160101)
os.makedirs("results/mean_rev_mc_final", exist_ok=True)

# DJIA 30 Mean Reversion — Asymmetric Long-Short  (Final — Monte Carlo on short-leg sizing with 80-trial Optuna best params)

**Iteration log:**
* **Pt1** — hand-picked regime-dependent short-leg sizing `(trending=0.25, volatile=1.0, crash=0.0)`. Result on 2022-2026 test: Sharpe 0.63, Sortino 0.91, CAGR 7.3%, MaxDD −14%.
* **Pt2** — residualized `ret_5`. **Worse** (Sharpe 0.51, CAGR 6.7%, MaxDD −17%); the raw-return signal was richer for the long leg in this regime. Abandoned.
* **Pt3** (this notebook) — replace the hand-picked sizing triple with a **300-trial Monte Carlo** on the short-leg regime sizing. Hold all 7 Pt1 Optuna best-params fixed; randomly sample `(s_trending, s_volatile, s_crash)` from wide uniform priors; select best by Sortino on the validation window; run final test-period backtest with the chosen triple. Goal: check whether Pt1's hand-picked sizing was actually optimal or whether a different region of the landscape dominates.

**Strategy thesis** (unchanged from Pt1). Equities drift upward. Short positions carry a structural headwind + unbounded-loss tail risk. Asymmetric sizing: 100% long + 20–40% short, regime-dependent. Long leg earns from oversold bounces; short leg earns only when dispersion is high.

**Architecture** (unchanged from Pt1). Same PIT scaffold, same XGBoost + Ridge + regime filter — just the sizing selection method changes from "hand-picked" to "MC-selected."

### Stage 1 â€” Data Collection (PIT constituent map)

In [4]:
# PIT-tracked DJIA 30 constituent map (verbatim from MomBased_Pt2).
# WBA excluded due to known yfinance data gaps; UTX was renamed to RTX before study start.

_BASELINE_APR2016 = {
    "AAPL", "AXP", "BA",  "CAT",  "CSCO", "CVX",  "DD",
    "DIS",  "GE",  "GS",  "HD",   "IBM",  "INTC", "JNJ",
    "JPM",  "KO",  "MCD", "MMM",  "MRK",  "MSFT", "NKE",
    "PFE",  "PG",  "RTX", "TRV",  "UNH",  "V",    "VZ",
    "WMT",  "XOM",
}

_CHANGES = [
    ("2018-06-26", ["WBA"],              ["GE"]),
    ("2019-04-02", ["DOW"],              ["DD"]),
    ("2020-04-06", ["RTX"],              ["UTX"]),
    ("2020-08-31", ["AMGN","CRM","HON"], ["XOM","PFE","RTX"]),
    ("2024-02-26", ["AMZN","SHW"],       ["WBA","INTC"]),
    ("2024-11-01", ["NVDA"],             ["DOW"]),
]


def fetch_prices_volumes(start_date="2016-04-01", end_date="2026-04-18"):
    """Download adjusted close AND volume â€” volumes needed for volume_z feature."""
    all_tickers = set(_BASELINE_APR2016)
    for _, added, removed in _CHANGES:
        all_tickers.update(added)
    all_tickers -= {"WBA", "UTX"}
    all_tickers = sorted(all_tickers)

    raw     = yf.download(all_tickers, start=start_date, end=end_date,
                          auto_adjust=True, progress=False)
    prices  = raw["Close"].ffill(limit=10)
    volumes = raw["Volume"].ffill(limit=10)
    prices  = prices.dropna(axis=1, how="all")
    volumes = volumes.reindex(columns=prices.columns)
    return prices, volumes


def get_constituents_on_date(date):
    """Return the exact DJIA 30 constituents on a given date."""
    constituents = set(_BASELINE_APR2016)
    for change_date_str, added, removed in _CHANGES:
        if date >= pd.Timestamp(change_date_str):
            constituents.update(added)
            constituents -= set(removed)
    constituents -= {"WBA", "UTX"}
    return constituents

### Stage 2 â€” Reversal Feature Engineering

Five short-horizon reversal features:

| Feature | Definition | Oversold signal |
|---|---|---|
| `ret_5` | trailing 5-day return | most negative |
| `rsi_2` | Connors-style short RSI | near 0 |
| `dist_bb` | (price âˆ’ 20d SMA) / (2 Ã— 20d std), Bollinger z-score | negative |
| `vol_20` | annualized 20-day realized vol | high (reversal edge strongest in high-dispersion names) |
| `volume_z` | today's volume z-score vs 20-day mean/std | high abs value (panic / frenzy marker) |

**Target**: 5-day forward return, cross-sectionally ranked to [0, 1]. **Non-overlap rule**: training rows subsampled every 5 days to avoid label overlap leakage on the 5-day horizon.

In [5]:
FEATURE_COLS = ["ret_5", "rsi_2", "dist_bb", "vol_20", "volume_z"]


def compute_rsi(prices_arr, period):
    """RSI from the last (period+1) closing prices. Same helper as MomBased_Pt2."""
    delta  = np.diff(prices_arr[-(period + 1):])
    gains  = delta[delta > 0].sum() / period
    losses = -delta[delta < 0].sum() / period
    if losses == 0:
        return 100.0
    return 100.0 - 100.0 / (1.0 + gains / losses)


def compute_features_reversal(
    prices,
    volumes,
    rsi_period=2,
    bb_window=20,
    vol_window=20,
    volume_window=20,
    fwd_days=5,
):
    """
    Build daily cross-sectional reversal feature snapshots.
    Returns long-format DataFrame with one row per (date, ticker).
    """
    log_rets = np.log(prices / prices.shift(1))
    min_day = max(fwd_days, bb_window, vol_window, volume_window, rsi_period + 1) + 2
    records = []

    for di in range(min_day, len(prices) - fwd_days):
        date   = prices.index[di]
        fwd_di = di + fwd_days

        pit_tickers = get_constituents_on_date(date)
        pit_tickers = [t for t in pit_tickers if t in prices.columns]

        row_data = []
        for t in pit_tickers:
            px  = prices[t].values
            vol = volumes[t].values

            if np.isnan(px[di]) or np.isnan(px[fwd_di]):
                continue
            if di - max(bb_window, vol_window, volume_window) < 0:
                continue

            p_back = px[di - fwd_days]
            if np.isnan(p_back) or p_back <= 0:
                continue
            ret_5 = px[di] / p_back - 1.0

            if di - rsi_period < 0:
                continue
            try:
                rsi = compute_rsi(px[: di + 1], rsi_period)
            except Exception:
                continue

            px_win = px[di - bb_window + 1 : di + 1]
            if np.any(np.isnan(px_win)):
                continue
            sma = px_win.mean()
            sd  = px_win.std()
            if sd <= 0:
                continue
            dist_bb = (px[di] - sma) / (2.0 * sd)

            r_slice = log_rets[t].iloc[di - vol_window + 1 : di + 1].values
            if np.any(np.isnan(r_slice)):
                continue
            vol_20 = float(np.std(r_slice)) * np.sqrt(252.0)

            v_win = vol[di - volume_window + 1 : di + 1]
            if np.any(np.isnan(v_win)) or v_win.std() <= 0:
                continue
            volume_z = (vol[di] - v_win.mean()) / v_win.std()

            fwd = px[fwd_di] / px[di] - 1.0

            row_data.append({
                "date": date, "ticker": t,
                "ret_5": ret_5, "rsi_2": rsi, "dist_bb": dist_bb,
                "vol_20": vol_20, "volume_z": volume_z,
                "fwd_ret": fwd,
            })

        df_row = pd.DataFrame(row_data)
        if df_row.empty:
            continue
        df_row["fwd_rank"] = df_row["fwd_ret"].rank(pct=True, na_option="keep")
        df_row = df_row.dropna(subset=["fwd_ret", "fwd_rank"] + FEATURE_COLS)
        if len(df_row) >= 5:
            records.append(df_row)

    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

### Stage 3 â€” Regime Classifier (verbatim from Pt2)

Three market states are derived from the rolling vol and drawdown of an equal-weight PIT index:

* **Trending** â€” normal markets (~75% of days)
* **Volatile** â€” elevated vol or moderate drawdown (~18%)
* **Crash** â€” extreme vol or deep drawdown (~7%)

The regime state drives the asymmetric sizing logic in the backtest.

In [6]:
def classify_regimes(
    prices,
    vol_window=20,
    dd_window=60,
    vol_crash=0.28,
    vol_vol=0.17,
    dd_crash=-0.13,
    dd_vol=-0.07,
):
    pit_index = []
    for date in prices.index:
        members = get_constituents_on_date(date)
        valid = [t for t in members if t in prices.columns and pd.notna(prices.loc[date, t])]
        pit_index.append(prices.loc[date, valid].mean() if valid else np.nan)
    mkt = pd.Series(pit_index, index=prices.index).ffill()
    log_rets = np.log(mkt / mkt.shift(1))
    roll_vol = log_rets.rolling(vol_window).std() * np.sqrt(252)
    roll_pk  = mkt.rolling(dd_window).max()
    roll_dd  = (mkt - roll_pk) / roll_pk

    regimes = pd.Series("trending", index=prices.index, dtype=str)
    regimes[(roll_vol > vol_vol)   | (roll_dd < dd_vol)]   = "volatile"
    regimes[(roll_vol > vol_crash) | (roll_dd < dd_crash)] = "crash"
    return regimes

### Stage 4 â€” XGBoost Cross-Sectional Ranker

Predicts cross-sectional forward rank from the 5 reversal features. Same architecture as Pt2 â€” regression on `fwd_rank` (0â€“1 percentile). Higher predicted rank â†’ expected bouncer; lower predicted rank â†’ expected pullback.

In [7]:
def train_xgboost(train_df, max_depth=3):
    clean = train_df[FEATURE_COLS + ["fwd_rank"]].replace([np.inf, -np.inf], np.nan).dropna()
    X = clean[FEATURE_COLS].values.astype(np.float32)
    y = clean["fwd_rank"].values.astype(np.float32)
    if len(X) == 0:
        raise ValueError("Training data is empty after cleaning.")

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=max_depth,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1.0,
        min_child_weight=5,
        reg_lambda=1.0,
        reg_alpha=0.1,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    model.fit(X, y)
    return model


def predict_scores(model, feat_df):
    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    return model.predict(X)

### Stage 5 â€” Asymmetric Long-Short Portfolio Construction

Two independent ridge solves:

**Long leg:**
$$ \max \sum \text{score}_i \cdot w_i - \lambda_{\text{long}} \|w\|^2 \quad \text{s.t.} \quad \sum w_i = \text{gross}_L, \; 0 \le w_i \le \text{cap}_L $$

**Short leg:**
$$ \max \sum (1 - \text{score}_i) \cdot w_i - \lambda_{\text{short}} \|w\|^2 \quad \text{s.t.} \quad \sum w_i = \text{gross}_S, \; 0 \le w_i \le \text{cap}_S $$

Short leg uses `(1 âˆ’ score)` as strength because we want more weight on names XGBoost expects to fall. Weights are then signed (+1 for long leg, âˆ’1 for short leg) and concatenated.

`Î»_short > Î»_long` by design â€” a single mis-identified short is a squeeze risk (unbounded loss); a single mis-identified long is bounded at âˆ’100%. Higher Î» on the short side forces breadth.

**Per-name caps**: 12% long, 6% short (halved) â€” another layer of short-tail defense.

In [8]:
def ridge_optimize_signed(
    scores, tickers,
    ridge_lambda=0.5,
    top_n=10,
    gross=1.0,
    direction=+1,
    pick="top",
    per_name_cap=None,
):
    """
    Ridge-regularised weight vector, scaled to target gross, signed by direction.
    """
    order = np.argsort(scores)
    sel_idx = order[::-1][:top_n] if pick == "top" else order[:top_n]

    sel_tickers = [tickers[i] for i in sel_idx]
    sel_scores  = scores[sel_idx]

    # For short leg, flip so low fwd_rank predictions get high "strength"
    strength = (1.0 - sel_scores) if direction < 0 else sel_scores

    raw_w = np.maximum(0.0, strength) / (2.0 * ridge_lambda)
    total = raw_w.sum()
    w_mag = raw_w / total if total > 0 else np.full(top_n, 1.0 / top_n)

    w_mag = w_mag * gross

    if per_name_cap is not None:
        w_mag = np.minimum(w_mag, per_name_cap)
        if w_mag.sum() > 0 and w_mag.sum() < gross:
            residual = gross - w_mag.sum()
            unused = np.maximum(0.0, per_name_cap - w_mag)
            if unused.sum() > 0:
                w_mag = w_mag + residual * unused / unused.sum()

    signed = w_mag * direction
    return dict(zip(sel_tickers, signed))


def asymmetric_long_short(
    scores, tickers,
    lambda_long=0.5, lambda_short=2.0,
    top_n_long=15, top_n_short=8,
    gross_long=1.0, gross_short=0.4,
    cap_long=0.12, cap_short=0.06,
    size_long=1.0, size_short=1.0,
):
    """
    Build asymmetric long-short book.
      LONG  leg: top_n_long picks (highest predicted fwd_rank)   = expected bouncers
      SHORT leg: top_n_short picks (lowest  predicted fwd_rank)  = expected pullbacks
    Regime multipliers (size_long, size_short) scale each leg's gross exposure.
    """
    w_long = ridge_optimize_signed(
        scores, tickers,
        ridge_lambda=lambda_long, top_n=top_n_long,
        gross=gross_long * size_long, direction=+1, pick="top",
        per_name_cap=cap_long,
    )
    w_short = ridge_optimize_signed(
        scores, tickers,
        ridge_lambda=lambda_short, top_n=top_n_short,
        gross=gross_short * size_short, direction=-1, pick="bottom",
        per_name_cap=cap_short,
    )
    combined = dict(w_long)
    for t, w in w_short.items():
        combined[t] = w   # short leg wins if a ticker somehow lands in both
    return combined

### Stage 6 â€” Backtest with Signed Weights, Borrow Cost, Regime-Aware Sizing

**Regime sizing** â€” asymmetric across regimes:

| Regime | size_long | size_short | Rationale |
|---|---|---|---|
| Trending (~75%) | 1.00Ã— | **0.25Ã—** | Long leg earns through drift; short leg fights the momentum factor |
| Volatile (~18%) | 1.00Ã— | **1.00Ã—** | High dispersion = reversal's home regime |
| Crash (~7%)    | 1.00Ã— | **0.00Ã—** | Shorts killed entirely â€” squeeze risk spikes in bear-market rallies |

**Costs:**
* Turnover: 10 bps Ã— `|Î”w|.sum()` on every rebalance (same as Pt2)
* Borrow: 25 bps/yr Ã— short notional, accrued daily

**Splits:** train 2016â€“20, val 2020â€“22 (Optuna sees this), test 2022â€“26 (held out).

In [9]:
REGIME_SIZING = {
    "trending": (1.00, 0.25),   # long full, short trimmed (fighting momentum factor)
    "volatile": (1.00, 1.00),   # both full (dispersion opportunity)
    "crash":    (1.00, 0.00),   # long-only (squeeze defense)
}

TURNOVER_BPS      = 10.0
BORROW_BPS_ANNUAL = 25.0


def sortino_ratio(daily_rets, mar=0.0):
    """Annualized Sortino ratio from daily returns."""
    excess = daily_rets - mar / 252.0
    downside = np.minimum(0.0, excess)
    dd_dev = np.sqrt(np.mean(downside ** 2))
    if dd_dev == 0:
        return float("inf") if excess.mean() > 0 else 0.0
    return (excess.mean() / dd_dev) * np.sqrt(252.0)


def run_backtest(
    prices, features_df, regimes,
    rsi_period=2, rebal_freq=1,
    lambda_long=0.5, lambda_short=2.0,
    top_n_long=15, top_n_short=8,
    gross_long=1.0, gross_short=0.4,
    cap_long=0.12, cap_short=0.06,
    mode="val",
    regime_sizing=None,
):
    # Pt3 addition: regime_sizing can be overridden per trial for MC sweeps
    sizing = regime_sizing if regime_sizing is not None else REGIME_SIZING
    rebal_days   = rebal_freq * 5
    unique_dates = np.sort(features_df["date"].unique())

    train_cutoff = pd.Timestamp("2020-01-01")
    val_cutoff   = pd.Timestamp("2022-01-01")

    train_dates = unique_dates[unique_dates <  train_cutoff]
    val_dates   = unique_dates[(unique_dates >= train_cutoff) & (unique_dates < val_cutoff)]
    test_dates  = unique_dates[unique_dates >= val_cutoff]

    eval_dates = val_dates if mode == "val" else test_dates

    if len(train_dates) < 20 or len(eval_dates) < 20:
        return {"sharpe": -99.0, "sortino": -99.0, "cagr": -99.0, "max_dd": -99.0}

    # Non-overlap rule: subsample training to every 5th day for 5-day forward target
    train_df_full = features_df[features_df["date"].isin(train_dates)]
    keep_train_dates = np.sort(train_df_full["date"].unique())[::5]
    train_df = train_df_full[train_df_full["date"].isin(keep_train_dates)]

    model   = train_xgboost(train_df)
    test_df = features_df[features_df["date"].isin(eval_dates)]

    test_start  = pd.Timestamp(eval_dates[0])
    test_prices = prices[prices.index >= test_start].copy()

    curr_weights = {}
    strat_val = 100.0
    bench_val = 100.0
    long_val  = 100.0
    short_val = 100.0
    peak = 100.0
    max_dd = 0.0
    kills_short = 0
    trim_trending = 0
    last_rebal = -rebal_days

    strat_curve, bench_curve, date_index, regime_log = [], [], [], []
    long_leg_curve, short_leg_curve = [], []

    for di in range(1, len(test_prices)):
        date = test_prices.index[di]

        pit_tickers = get_constituents_on_date(date)
        pit_tickers = [t for t in pit_tickers if t in test_prices.columns]

        daily_rets = {}
        for t in pit_tickers:
            p0 = test_prices[t].iloc[di - 1]
            p1 = test_prices[t].iloc[di]
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                daily_rets[t] = p1 / p0 - 1.0

        reg = regimes.get(date, "trending")
        regime_log.append(reg)

        if di - last_rebal >= rebal_days:
            last_rebal = di
            avail = test_df[test_df["date"] <= date]
            if not avail.empty:
                snap_date = avail["date"].max()
                snap = (
                    test_df[(test_df["date"] == snap_date) &
                            (test_df["ticker"].isin(pit_tickers))]
                    .set_index("ticker")
                    .dropna(subset=FEATURE_COLS)
                )

                if len(snap) >= top_n_long + top_n_short:
                    scores_arr = predict_scores(model, snap.reset_index())
                    tickers_avail = snap.index.tolist()

                    size_long_mult, size_short_mult = sizing[reg]
                    if reg == "crash":
                        kills_short += 1
                    if reg == "trending":
                        trim_trending += 1

                    new_weights = asymmetric_long_short(
                        scores_arr, tickers_avail,
                        lambda_long=lambda_long, lambda_short=lambda_short,
                        top_n_long=top_n_long, top_n_short=top_n_short,
                        gross_long=gross_long, gross_short=gross_short,
                        cap_long=cap_long, cap_short=cap_short,
                        size_long=size_long_mult, size_short=size_short_mult,
                    )

                    # Turnover cost on |Î”w|
                    all_t = set(new_weights) | set(curr_weights)
                    turnover = sum(abs(new_weights.get(t, 0.0) - curr_weights.get(t, 0.0))
                                   for t in all_t)
                    cost = turnover * TURNOVER_BPS / 10000.0
                    strat_val *= (1.0 - cost)
                    curr_weights = new_weights

        # P&L â€” signed weights
        strat_ret = sum(curr_weights.get(t, 0.0) * daily_rets.get(t, 0.0)
                        for t in curr_weights if t in daily_rets)
        long_ret  = sum(w * daily_rets.get(t, 0.0)
                        for t, w in curr_weights.items()
                        if w > 0 and t in daily_rets)
        short_ret = sum(w * daily_rets.get(t, 0.0)
                        for t, w in curr_weights.items()
                        if w < 0 and t in daily_rets)

        # Daily borrow cost on short notional
        short_notional    = sum(abs(w) for w in curr_weights.values() if w < 0)
        borrow_cost_daily = short_notional * (BORROW_BPS_ANNUAL / 10000.0) / 252.0
        strat_ret -= borrow_cost_daily
        short_ret -= borrow_cost_daily

        bench_ret = float(np.mean([daily_rets[t] for t in pit_tickers if t in daily_rets]))

        strat_val *= (1.0 + strat_ret)
        bench_val *= (1.0 + bench_ret)
        long_val  *= (1.0 + long_ret)
        short_val *= (1.0 + short_ret)
        peak      = max(peak, strat_val)
        max_dd    = min(max_dd, (strat_val - peak) / peak)

        strat_curve.append(strat_val)
        bench_curve.append(bench_val)
        long_leg_curve.append(long_val)
        short_leg_curve.append(short_val)
        date_index.append(date)

    if len(strat_curve) < 50:
        return {"sharpe": -99.0, "sortino": -99.0, "cagr": -99.0, "max_dd": -99.0}

    daily_rets_arr = np.diff(strat_curve) / np.array(strat_curve[:-1])
    n_years    = len(strat_curve) / 252.0
    cagr       = (strat_val / 100.0) ** (1.0 / n_years) - 1.0
    bench_cagr = (bench_val / 100.0) ** (1.0 / n_years) - 1.0
    sharpe  = ((daily_rets_arr.mean() / daily_rets_arr.std()) * np.sqrt(252.0)
               if daily_rets_arr.std() > 0 else 0.0)
    sortino = sortino_ratio(daily_rets_arr)

    feat_imp = dict(zip(FEATURE_COLS, model.feature_importances_))

    return {
        "sharpe":          round(float(sharpe),  4),
        "sortino":         round(float(sortino), 4),
        "cagr":            round(float(cagr * 100), 2),
        "bench_cagr":      round(float(bench_cagr * 100), 2),
        "max_dd":          round(float(max_dd * 100), 2),
        "cum_ret":         round(strat_val - 100.0, 2),
        "kills_short":     kills_short,
        "trim_trending":   trim_trending,
        "strat_curve":     strat_curve,
        "bench_curve":     bench_curve,
        "long_leg_curve":  long_leg_curve,
        "short_leg_curve": short_leg_curve,
        "date_index":      date_index,
        "regime_log":      regime_log,
        "feat_imp":        feat_imp,
        "model":           model,
        "weights":         curr_weights,
    }

### Stage 7 — Monte Carlo Short-Leg Sizing Sweep (Pt3 replacement for Optuna)

Pt1 chose the short-leg regime sizing by hand: `(trending=0.25, volatile=1.0, crash=0.0)`. The reasoning was sound (shorts fight the momentum factor in trending regimes, carry squeeze risk in crashes), but it's one point on a 3-dimensional surface. Pt3 maps the surface.

**Experiment design:**
* **300 random trials**. Each trial samples a `(s_trending, s_volatile, s_crash)` triple from wide uniform priors:
  * `s_trending ~ Uniform[0.0, 1.0]` — fraction of `gross_short` used on trending days
  * `s_volatile ~ Uniform[0.0, 1.5]` — may exceed 1.0 (lever up in the signal's home regime)
  * `s_crash ~ Uniform[0.0, 0.5]` — small window for keeping shorts on in crashes
* **All 7 Pt1 Optuna best-params held fixed** — clean A/B test isolating the sizing effect.
* **Selection on validation** (2020-2022), then **final backtest on held-out test** (2022-2026) with the MC-winning sizing.
* Objective: **Sortino ratio** (same as Pt1's Optuna).

This gives us both (a) a point estimate of the best sizing and (b) a **distribution** of outcomes — useful for the ICM's "conditions under which it loses money" section.

In [10]:
# Pt1 best-params (frozen — clean A/B test on sizing)
PT1_BEST = {
    # From MeanRev-AsymLs-Final.ipynb (80 Optuna trials)
    "rsi_period":   7,
    "rebal_freq":   2,
    "lambda_long":  0.9116238143505125,
    "lambda_short": 3.503138099317926,
    "top_n_long":   15,
    "top_n_short":  12,
    "gross_short":  0.20,
    "gross_long":   1.0,
    "cap_long":     0.12,
    "cap_short":    0.06,
}


N_MC_TRIALS = 300


def monte_carlo_sizing(
    prices, volumes, regimes, feat_df,
    base_params=PT1_BEST,
    n_trials=N_MC_TRIALS,
    s_trending_range=(0.0, 1.0),
    s_volatile_range=(0.0, 1.5),
    s_crash_range=(0.0, 0.5),
    seed=42,
    mode="val",
):
    """
    Random-uniform sweep over short-leg regime sizing multipliers.

    For each trial, samples (s_trending, s_volatile, s_crash) and runs
    run_backtest with Pt1's best 7 params plus the random sizing.
    Returns a DataFrame with one row per trial.
    """
    rng = np.random.default_rng(seed)
    records = []
    print(f"\n{'='*60}\n  Monte Carlo — {n_trials} trials, short-leg sizing\n{'='*60}")

    for i in range(n_trials):
        s_trend = float(rng.uniform(*s_trending_range))
        s_vol   = float(rng.uniform(*s_volatile_range))
        s_crash = float(rng.uniform(*s_crash_range))

        sizing = {
            "trending": (1.00, s_trend),
            "volatile": (1.00, s_vol),
            "crash":    (1.00, s_crash),
        }

        result = run_backtest(
            prices, feat_df, regimes,
            **base_params,
            mode=mode,
            regime_sizing=sizing,
        )

        records.append({
            "trial": i,
            "s_trending": s_trend,
            "s_volatile": s_vol,
            "s_crash":    s_crash,
            "sharpe":  result.get("sharpe",  -99),
            "sortino": result.get("sortino", -99),
            "cagr":    result.get("cagr",    -99),
            "max_dd":  result.get("max_dd",  -99),
            "cum_ret": result.get("cum_ret", -99),
        })

        if (i + 1) % 25 == 0 or i == n_trials - 1:
            best_so_far = max(r["sortino"] for r in records)
            print(f"  Trial {i+1:3d}/{n_trials}  "
                  f"(s_trend={s_trend:.2f}, s_vol={s_vol:.2f}, s_crash={s_crash:.2f})  "
                  f"Sortino={records[-1]['sortino']:+.3f}  best={best_so_far:+.3f}")

    return pd.DataFrame(records)

### Stage 8 â€” Dashboard Plot

In [11]:
DARK = "#0a0c0f"; SURFACE = "#111418"; BORDER = "#232830"
TEXT = "#e2e8f0"; MUTED = "#8896a8"
GREEN = "#22c55e"; RED = "#ef4444"; AMBER = "#f59e0b"
BLUE = "#60a5fa"; PURPLE = "#a78bfa"; CYAN = "#22d3ee"

plt.rcParams.update({
    "figure.facecolor": DARK,    "axes.facecolor":  SURFACE,
    "axes.edgecolor":   BORDER,  "axes.labelcolor": MUTED,
    "xtick.color":      MUTED,   "ytick.color":     MUTED,
    "text.color":       TEXT,    "grid.color":      BORDER,
    "grid.linewidth":   0.5,     "font.family":     "monospace",
    "axes.titlecolor":  TEXT,    "axes.titlesize":  10,
    "axes.titleweight": "bold",
})


def plot_backtest(result, best_params, save_path):
    fig = plt.figure(figsize=(16, 12), facecolor=DARK)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35,
                            left=0.07, right=0.97, top=0.92, bottom=0.07)

    dates   = pd.DatetimeIndex(result["date_index"])
    strat   = np.array(result["strat_curve"])
    bench   = np.array(result["bench_curve"])
    long_c  = np.array(result["long_leg_curve"])
    short_c = np.array(result["short_leg_curve"])
    regimes = result["regime_log"]

    reg_colors = {"trending": GREEN, "volatile": AMBER, "crash": RED}

    # Panel 1: cumulative performance
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(dates, strat, color=CYAN, lw=2.0,
             label=f"Strategy   CAGR {result['cagr']:.1f}%", zorder=3)
    ax1.plot(dates, bench, color=BLUE, lw=1.4, ls="--", alpha=0.7,
             label=f"Benchmark  CAGR {result['bench_cagr']:.1f}%", zorder=2)

    prev_reg, seg_start = regimes[0], dates[0]
    for i in range(1, len(dates)):
        if regimes[i] != prev_reg or i == len(dates) - 1:
            ax1.axvspan(seg_start, dates[i], alpha=0.09,
                        color=reg_colors[prev_reg], zorder=1)
            seg_start = dates[i]; prev_reg = regimes[i]

    legend_patches = [
        Patch(facecolor=GREEN, alpha=0.4, label="Trending (L 1.0x, S 0.25x)"),
        Patch(facecolor=AMBER, alpha=0.4, label="Volatile (L 1.0x, S 1.0x)"),
        Patch(facecolor=RED,   alpha=0.4, label="Crash (L 1.0x, S OFF)"),
    ]
    ax1.legend(handles=[*ax1.get_lines(), *legend_patches],
               loc="upper left", fontsize=8.5,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT, ncol=2)
    ax1.set_title("CUMULATIVE PERFORMANCE  (shaded = regime)", pad=8)
    ax1.set_ylabel("Portfolio value  (base = 100)")
    ax1.grid(True, alpha=0.3)

    # Panel 2: drawdown
    ax2 = fig.add_subplot(gs[1, :2])
    peak_arr = np.maximum.accumulate(strat)
    dd_arr   = (strat - peak_arr) / peak_arr * 100.0
    ax2.fill_between(dates, dd_arr, 0, color=RED, alpha=0.45)
    ax2.plot(dates, dd_arr, color=RED, lw=0.8)
    ax2.set_title(f"DRAWDOWN  (max {result['max_dd']:.1f}%)")
    ax2.set_ylabel("%")
    ax2.grid(True, alpha=0.3)

    # Panel 3: feature importance
    ax3 = fig.add_subplot(gs[1, 2])
    feat_imp = result["feat_imp"]
    labels = ["ret_5", f"rsi_{best_params['rsi_period']}", "dist_bb",
              "vol_20", "volume_z"]
    vals = [float(feat_imp.get(f, 0.0)) for f in FEATURE_COLS]
    total = sum(vals) if sum(vals) > 0 else 1.0
    vals_pct = np.array(vals) / total * 100.0
    colors_fi = [GREEN, BLUE, PURPLE, AMBER, CYAN]
    bars = ax3.barh(labels, vals_pct, color=colors_fi, height=0.6)
    for bar, v in zip(bars, vals_pct):
        ax3.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
                 f"{v:.1f}%", va="center", fontsize=9, color=TEXT)
    ax3.set_title("XGBOOST FEATURE IMPORTANCE")
    ax3.set_xlim(0, max(vals_pct) * 1.3)
    ax3.grid(True, alpha=0.3, axis="x")

    # Panel 4: leg attribution
    ax4 = fig.add_subplot(gs[2, :2])
    long_ret  = (long_c[-1]  / long_c[0]  - 1) * 100
    short_ret = (short_c[-1] / short_c[0] - 1) * 100
    ax4.plot(dates, long_c, color=GREEN, lw=1.6,
             label=f"Long leg   {long_ret:+.1f}%  (oversold bouncers)")
    ax4.plot(dates, short_c, color=RED, lw=1.6,
             label=f"Short leg  {short_ret:+.1f}%  (overbought pullbacks)")
    ax4.axhline(100.0, color=MUTED, ls=":", lw=0.8, alpha=0.6)
    ax4.set_title("LEG ATTRIBUTION  (each leg's standalone P&L)")
    ax4.set_ylabel("Leg value  (base = 100)")
    ax4.legend(loc="upper left", fontsize=8.5,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax4.grid(True, alpha=0.3)

    # Panel 5: metrics + best params
    ax5 = fig.add_subplot(gs[2, 2])
    ax5.axis("off")
    rows = [
        ("Ann. return",    f"{result['cagr']:.2f}%",       CYAN),
        ("Benchmark",      f"{result['bench_cagr']:.2f}%", BLUE),
        ("Sharpe ratio",   f"{result['sharpe']:.4f}",
         GREEN if result["sharpe"] > 0.5 else AMBER),
        ("Sortino ratio",  f"{result['sortino']:.4f}",
         GREEN if result["sortino"] > 0.7 else AMBER),
        ("Max drawdown",   f"{result['max_dd']:.2f}%",     RED),
        ("Cumul. return",  f"{result['cum_ret']:.1f}%",
         GREEN if result["cum_ret"] > 0 else RED),
        ("Short-kill days",f"{result['kills_short']}",     PURPLE),
        ("-- best params --", "", MUTED),
        ("RSI period",     f"{best_params['rsi_period']}d",TEXT),
        ("Rebal freq",     f"{best_params['rebal_freq']}w",TEXT),
        ("lambda long",    f"{best_params['lambda_long']:.3f}",  TEXT),
        ("lambda short",   f"{best_params['lambda_short']:.3f}", TEXT),
        ("Top-N long",     f"{best_params['top_n_long']}",  TEXT),
        ("Top-N short",    f"{best_params['top_n_short']}", TEXT),
        ("Gross short",    f"{best_params['gross_short']:.2f}",  TEXT),
    ]
    for i, (label, val, color) in enumerate(rows):
        y = 1.0 - i * 0.065
        ax5.text(0.02, y, label, transform=ax5.transAxes,
                 fontsize=9, color=MUTED, va="top")
        ax5.text(0.98, y, val, transform=ax5.transAxes,
                 fontsize=9, color=color, va="top", ha="right", fontweight="bold")

    fig.suptitle(
        "DOW 30 MEAN REVERSION  //  Asymmetric Long-Short  //  XGBoost + Ridge + Regime",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.97
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  Saved: {save_path}")
    plt.close()

def plot_mc_landscape(mc_df, pt1_sizing, pt1_sortino, save_path):
    """
    6-panel Monte Carlo dashboard:
      1. scatter s_trending vs s_volatile, colored by Sortino
      2. scatter s_trending vs s_crash, colored by Sortino
      3. scatter s_volatile vs s_crash, colored by Sortino
      4-6. histograms of Sortino, CAGR, MaxDD across all trials
    Pt1's hand-picked sizing is marked with a cross; its Sortino is a vertical line.
    """
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("rg", [RED, AMBER, GREEN])

    fig = plt.figure(figsize=(16, 10), facecolor=DARK)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.30,
                            left=0.07, right=0.97, top=0.92, bottom=0.08)

    sortino = mc_df["sortino"].values
    vmin = np.nanpercentile(sortino, 5)
    vmax = np.nanpercentile(sortino, 95)

    # Best trial
    best = mc_df.loc[mc_df["sortino"].idxmax()]

    def _scatter(ax, x, y, c, xlabel, ylabel, pt1_xy):
        sc = ax.scatter(x, y, c=c, cmap=cmap, vmin=vmin, vmax=vmax,
                        s=24, alpha=0.75, edgecolors="none")
        # Mark best
        ax.scatter([best[xlabel]], [best[ylabel]], s=180, facecolors="none",
                   edgecolors=CYAN, linewidths=2.0, label=f"MC best ({best[xlabel]:.2f}, {best[ylabel]:.2f})")
        # Mark Pt1
        ax.scatter([pt1_xy[0]], [pt1_xy[1]], s=180, marker="X",
                   color=BLUE, edgecolors=TEXT, linewidths=1.0,
                   label=f"Pt1 ({pt1_xy[0]}, {pt1_xy[1]})")
        ax.set_xlabel(xlabel, color=MUTED)
        ax.set_ylabel(ylabel, color=MUTED)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7.5, facecolor=SURFACE, edgecolor=BORDER,
                  labelcolor=TEXT, loc="best")
        return sc

    # Panel 1: s_trending vs s_volatile
    ax = fig.add_subplot(gs[0, 0])
    sc = _scatter(ax, mc_df["s_trending"], mc_df["s_volatile"], sortino,
                  "s_trending", "s_volatile", (pt1_sizing[0], pt1_sizing[1]))
    ax.set_title("s_trending vs s_volatile  (color = Sortino)", color=TEXT)
    plt.colorbar(sc, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    # Panel 2: s_trending vs s_crash
    ax = fig.add_subplot(gs[0, 1])
    sc = _scatter(ax, mc_df["s_trending"], mc_df["s_crash"], sortino,
                  "s_trending", "s_crash", (pt1_sizing[0], pt1_sizing[2]))
    ax.set_title("s_trending vs s_crash  (color = Sortino)", color=TEXT)
    plt.colorbar(sc, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    # Panel 3: s_volatile vs s_crash
    ax = fig.add_subplot(gs[0, 2])
    sc = _scatter(ax, mc_df["s_volatile"], mc_df["s_crash"], sortino,
                  "s_volatile", "s_crash", (pt1_sizing[1], pt1_sizing[2]))
    ax.set_title("s_volatile vs s_crash  (color = Sortino)", color=TEXT)
    plt.colorbar(sc, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    # Panel 4: Sortino histogram
    ax = fig.add_subplot(gs[1, 0])
    ax.hist(sortino, bins=30, color=GREEN, edgecolor=BORDER, alpha=0.75)
    ax.axvline(pt1_sortino, color=BLUE, ls="--", lw=1.5, label=f"Pt1 val Sortino {pt1_sortino:.3f}")
    ax.axvline(best["sortino"], color=CYAN, ls="--", lw=1.5, label=f"MC best Sortino {best['sortino']:.3f}")
    ax.set_xlabel("Sortino (validation)", color=MUTED)
    ax.set_ylabel("Trial count", color=MUTED)
    ax.set_title("VALIDATION SORTINO DISTRIBUTION", color=TEXT)
    ax.legend(fontsize=8, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3, axis="y")

    # Panel 5: CAGR histogram
    ax = fig.add_subplot(gs[1, 1])
    ax.hist(mc_df["cagr"].values, bins=30, color=AMBER, edgecolor=BORDER, alpha=0.75)
    ax.axvline(best["cagr"], color=CYAN, ls="--", lw=1.5, label=f"MC best {best['cagr']:.2f}%")
    ax.set_xlabel("CAGR % (validation)", color=MUTED)
    ax.set_ylabel("Trial count", color=MUTED)
    ax.set_title("VALIDATION CAGR DISTRIBUTION", color=TEXT)
    ax.legend(fontsize=8, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3, axis="y")

    # Panel 6: MaxDD histogram
    ax = fig.add_subplot(gs[1, 2])
    ax.hist(mc_df["max_dd"].values, bins=30, color=RED, edgecolor=BORDER, alpha=0.75)
    ax.axvline(best["max_dd"], color=CYAN, ls="--", lw=1.5, label=f"MC best {best['max_dd']:.2f}%")
    ax.set_xlabel("Max drawdown % (validation)", color=MUTED)
    ax.set_ylabel("Trial count", color=MUTED)
    ax.set_title("VALIDATION MAX DD DISTRIBUTION", color=TEXT)
    ax.legend(fontsize=8, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3, axis="y")

    n_better = int((sortino > pt1_sortino).sum())
    pct_better = 100.0 * n_better / len(sortino)
    fig.suptitle(
        f"MONTE CARLO — SHORT-LEG REGIME SIZING  //  {len(mc_df)} trials  //  "
        f"{n_better} ({pct_better:.0f}%) beat Pt1 val Sortino",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.96,
    )

    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  Saved: {save_path}")
    plt.close()


### Main Pipeline

In [12]:
def main():
    print("\n" + "="*60)
    print("  DOW 30 MEAN REVERSION  —  ASYMMETRIC L/S  (Pt3: MC sizing)")
    print("="*60)

    print("\n[1/5]  Downloading 10Y of daily prices + volumes ...")
    prices, volumes = fetch_prices_volumes()
    print(f"       {len(prices)} trading days x {len(prices.columns)} tickers")

    print("\n[2/5]  Classifying regimes ...")
    regimes = classify_regimes(prices)
    for s in ["trending", "volatile", "crash"]:
        n = (regimes == s).sum()
        print(f"       {s:10s}: {n:4d} days  ({n/len(regimes)*100:.1f}%)")

    print("\n[3/5]  Computing reversal features (rsi_period=2, fwd_5d) ...")
    feat_df = compute_features_reversal(
        prices, volumes,
        rsi_period=PT1_BEST["rsi_period"],
        fwd_days=5,
    )
    print(f"       {len(feat_df)} feature rows, {feat_df['date'].nunique()} unique dates")

    print(f"\n[4/5]  Running Monte Carlo sweep ({N_MC_TRIALS} trials) ...")
    mc_df = monte_carlo_sizing(
        prices, volumes, regimes, feat_df,
        base_params=PT1_BEST,
        n_trials=N_MC_TRIALS,
        mode="val",
    )
    mc_df.to_csv("results/mean_rev_mc_final/mc_trials.csv", index=False)

    # Also score Pt1's hand-picked sizing on the SAME validation period for fair comparison
    pt1_sizing_tuple = (0.25, 1.00, 0.00)
    pt1_sizing_dict = {
        "trending": (1.00, 0.25),
        "volatile": (1.00, 1.00),
        "crash":    (1.00, 0.00),
    }
    pt1_val = run_backtest(
        prices, feat_df, regimes,
        **PT1_BEST,
        mode="val",
        regime_sizing=pt1_sizing_dict,
    )
    pt1_val_sortino = pt1_val["sortino"]

    best = mc_df.loc[mc_df["sortino"].idxmax()]
    best_sizing = (float(best["s_trending"]), float(best["s_volatile"]), float(best["s_crash"]))
    n_better = int((mc_df["sortino"] > pt1_val_sortino).sum())
    pct_better = 100.0 * n_better / len(mc_df)

    print(f"\n       Pt1 val-period Sortino (hand-picked sizing):   {pt1_val_sortino:.4f}")
    print(f"       MC best val-period Sortino:                    {best['sortino']:.4f}")
    print(f"       Best sizing (s_trend, s_vol, s_crash):         ({best_sizing[0]:.3f}, {best_sizing[1]:.3f}, {best_sizing[2]:.3f})")
    print(f"       Trials beating Pt1: {n_better}/{len(mc_df)}  ({pct_better:.1f}%)")

    print("\n[5/5]  Final TEST-period backtest with MC-best sizing ...")
    best_sizing_dict = {
        "trending": (1.00, best_sizing[0]),
        "volatile": (1.00, best_sizing[1]),
        "crash":    (1.00, best_sizing[2]),
    }
    result = run_backtest(
        prices, feat_df, regimes,
        **PT1_BEST,
        mode="test",
        regime_sizing=best_sizing_dict,
    )
    lc = result["long_leg_curve"]
    sc = result["short_leg_curve"]

    print(f"\n       -- Final test-period metrics (2022-2026) --")
    print(f"       Ann. return  : {result['cagr']:.2f}%  (benchmark {result['bench_cagr']:.2f}%)")
    print(f"       Sharpe ratio : {result['sharpe']:.4f}")
    print(f"       Sortino ratio: {result['sortino']:.4f}")
    print(f"       Max drawdown : {result['max_dd']:.2f}%")
    print(f"       Cumulative   : {result['cum_ret']:.1f}%")
    print(f"       Long  leg    : {(lc[-1]/lc[0]-1)*100:+.1f}%")
    print(f"       Short leg    : {(sc[-1]/sc[0]-1)*100:+.1f}%")

    print("\n       Generating charts ...")
    globals()["result"] = result
    globals()["mc_df"] = mc_df
    plot_mc_landscape(mc_df, pt1_sizing_tuple, pt1_val_sortino,
                      "results/mean_rev_mc_final/mc_landscape.png")

    # Inject best_sizing and best MC params into dict so existing plot_backtest works
    best_params_for_plot = dict(PT1_BEST)
    best_params_for_plot.update({
        "rsi_period":  PT1_BEST["rsi_period"],
        "rebal_freq":  PT1_BEST["rebal_freq"],
        "gross_short": PT1_BEST["gross_short"],
    })
    plot_backtest(result, best_params_for_plot, "results/mean_rev_mc_final/backtest_summary.png")

    # Save tidy summary
    with open("results/mean_rev_mc_final/best_sizing.txt", "w") as f:
        f.write("MONTE CARLO BEST SHORT-LEG SIZING\n")
        f.write("=" * 40 + "\n")
        f.write(f"{'s_trending':15s}: {best_sizing[0]:.4f}\n")
        f.write(f"{'s_volatile':15s}: {best_sizing[1]:.4f}\n")
        f.write(f"{'s_crash':15s}: {best_sizing[2]:.4f}\n")
        f.write(f"\n")
        f.write(f"Pt1 sizing (hand-picked):         (0.25, 1.00, 0.00)\n")
        f.write(f"Pt1 val-period Sortino:           {pt1_val_sortino:.4f}\n")
        f.write(f"MC best val-period Sortino:       {best['sortino']:.4f}\n")
        f.write(f"MC trials beating Pt1 val:        {n_better}/{len(mc_df)}  ({pct_better:.1f}%)\n")
        f.write(f"\n")
        f.write(f"FROZEN PT1 OPTUNA PARAMS\n")
        f.write("=" * 40 + "\n")
        for k, v in PT1_BEST.items():
            f.write(f"{k:15s}: {v}\n")
        f.write(f"\n")
        f.write(f"TEST-PERIOD METRICS (2022-2026, held out)\n")
        f.write("=" * 40 + "\n")
        f.write(f"{'sharpe':20s}: {result['sharpe']:.4f}\n")
        f.write(f"{'sortino':20s}: {result['sortino']:.4f}\n")
        f.write(f"{'cagr_%':20s}: {result['cagr']:.2f}\n")
        f.write(f"{'bench_cagr_%':20s}: {result['bench_cagr']:.2f}\n")
        f.write(f"{'max_drawdown_%':20s}: {result['max_dd']:.2f}\n")
        f.write(f"{'cumulative_%':20s}: {result['cum_ret']:.2f}\n")
        f.write(f"{'long_leg_%':20s}: {(lc[-1]/lc[0]-1)*100:.2f}\n")
        f.write(f"{'short_leg_%':20s}: {(sc[-1]/sc[0]-1)*100:.2f}\n")
    print("       Saved: results/mean_rev_mc_final/best_sizing.txt")

    result_clean = {k: v for k, v in result.items() if k != "model"}
    with open("results/mean_rev_mc_final/result.pkl", "wb") as f:
        pickle.dump({
            "result": result_clean,
            "best_sizing": best_sizing,
            "mc_df": mc_df,
            "pt1_val_sortino": pt1_val_sortino,
        }, f)
    print("       Saved: results/mean_rev_mc_final/result.pkl")

    print(f"\n{'='*60}\n  Done. Outputs in results/mean_rev_mc_final/\n{'='*60}\n")
    return result, mc_df, best_sizing, prices, regimes


if __name__ == "__main__":
    result, mc_df, best_sizing, prices, regimes = main()


  DOW 30 MEAN REVERSION  —  ASYMMETRIC L/S  (Pt3: MC sizing)

[1/5]  Downloading 10Y of daily prices + volumes ...


       2526 trading days x 37 tickers

[2/5]  Classifying regimes ...


       trending  : 1895 days  (75.0%)
       volatile  :  451 days  (17.9%)
       crash     :  180 days  (7.1%)

[3/5]  Computing reversal features (rsi_period=2, fwd_5d) ...


       73534 feature rows, 2499 unique dates

[4/5]  Running Monte Carlo sweep (300 trials) ...

  Monte Carlo — 300 trials, short-leg sizing


  Trial  25/300  (s_trend=0.85, s_vol=0.35, s_crash=0.03)  Sortino=+0.911  best=+0.972


...

  Trial 300/300  (s_trend=0.20, s_vol=0.28, s_crash=0.26)  Sortino=+0.943  best=+0.983



       Pt1 val-period Sortino (hand-picked sizing):   0.8559
       MC best val-period Sortino:                    0.9825
       Best sizing (s_trend, s_vol, s_crash):         (0.004, 0.025, 0.497)
       Trials beating Pt1: 236/300  (78.7%)

[5/5]  Final TEST-period backtest with MC-best sizing ...



       -- Final test-period metrics (2022-2026) --
       Ann. return  : 12.07%  (benchmark 10.94%)
       Sharpe ratio : 0.8202
       Sortino ratio: 1.2041
       Max drawdown : -19.60%
       Cumulative   : 62.6%
       Long  leg    : +82.1%
       Short leg    : -0.3%

       Generating charts ...


  Saved: results/mean_rev_mc_final/mc_landscape.png


  Saved: results/mean_rev_mc_final/backtest_summary.png
       Saved: results/mean_rev_mc_final/best_sizing.txt
       Saved: results/mean_rev_mc_final/result.pkl

  Done. Outputs in results/mean_rev_mc_final/



### Diagnostics

In [13]:
# Run after main() completes
strat = np.array(result["strat_curve"])
bench = np.array(result["bench_curve"])
long_c  = np.array(result["long_leg_curve"])
short_c = np.array(result["short_leg_curve"])
dates = pd.DatetimeIndex(result["date_index"])

rets_s = np.diff(strat) / strat[:-1]
rets_b = np.diff(bench) / bench[:-1]
rets_l = np.diff(long_c) / long_c[:-1]
rets_sh = np.diff(short_c) / short_c[:-1]

print(f"Strategy  â€” ann vol: {rets_s.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_s.mean()*252*100:.2f}%")
print(f"Benchmark â€” ann vol: {rets_b.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_b.mean()*252*100:.2f}%")
print(f"Long leg  â€” ann vol: {rets_l.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_l.mean()*252*100:.2f}%")
print(f"Short leg â€” ann vol: {rets_sh.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_sh.mean()*252*100:.2f}%")
print()
print(f"Correlation strat vs bench:   {np.corrcoef(rets_s, rets_b)[0,1]:+.3f}")
print(f"Correlation long  vs bench:   {np.corrcoef(rets_l, rets_b)[0,1]:+.3f}")
print(f"Correlation short vs bench:   {np.corrcoef(rets_sh, rets_b)[0,1]:+.3f}")

df_rets = pd.DataFrame({"strat": rets_s, "bench": rets_b,
                        "long": rets_l, "short": rets_sh}, index=dates[1:])
annual = df_rets.resample("YE").apply(lambda x: (1+x).prod()-1) * 100
print("\nYear-by-year returns (%):")
print(annual.round(2).to_string())

Strategy  â€” ann vol: 15.4%   mean daily ret: 12.59%
Benchmark â€” ann vol: 14.8%   mean daily ret: 11.31%
Long leg  â€” ann vol: 15.4%   mean daily ret: 15.26%
Short leg â€” ann vol: 0.3%   mean daily ret: -0.08%

Correlation strat vs bench:   +0.951
Correlation long  vs bench:   +0.953
Correlation short vs bench:   -0.341

Year-by-year returns (%):
            strat  bench   long  short
2022-12-31  -8.88  -7.61  -6.67  -0.05
2023-12-31  21.84  18.28  25.07  -0.01
2024-12-31  15.03  17.06  18.16  -0.05
2025-12-31  20.96  16.70  24.40  -0.20
2026-12-31   5.30   3.51   6.13  -0.03


In [14]:
# Final weight composition on last rebalance
w = result["weights"]
long_w  = sorted([(t, x) for t, x in w.items() if x > 0], key=lambda x: -x[1])
short_w = sorted([(t, x) for t, x in w.items() if x < 0], key=lambda x: x[1])

print(f"Last rebalance composition:")
print(f"  {len(long_w)} longs totaling  {sum(x for _, x in long_w):+.3f}")
print(f"  {len(short_w)} shorts totaling {sum(x for _, x in short_w):+.3f}")
print(f"  Net exposure              {sum(x for _, x in long_w + short_w):+.3f}")
print(f"  Gross exposure            {sum(abs(x) for _, x in long_w + short_w):+.3f}")
print()
print("LONGS (oversold bouncers):")
for t, x in long_w:
    bar = "#" * int(x * 200)
    print(f"  {t:6s}  {x*100:+5.2f}%  {bar}")
print()
print("SHORTS (overbought pullbacks):")
for t, x in short_w:
    bar = "#" * int(abs(x) * 200)
    print(f"  {t:6s}  {x*100:+5.2f}%  {bar}")

Last rebalance composition:
  15 longs totaling  +1.000
  12 shorts totaling -0.005
  Net exposure              +0.995
  Gross exposure            +1.005

LONGS (oversold bouncers):
  CRM     +7.39%  ##############
  MMM     +7.13%  ##############
  DIS     +7.01%  ##############
  HD      +6.78%  #############
  AAPL    +6.78%  #############
  WMT     +6.63%  #############
  V       +6.61%  #############
  CSCO    +6.61%  #############
  KO      +6.60%  #############
  MCD     +6.53%  #############
  AMGN    +6.43%  ############
  AXP     +6.43%  ############
  MRK     +6.38%  ############
  MSFT    +6.38%  ############
  VZ      +6.31%  ############

SHORTS (overbought pullbacks):
  NKE     -0.05%  
  NVDA    -0.04%  
  HON     -0.04%  
  UNH     -0.04%  
  AMZN    -0.04%  
  CAT     -0.04%  
  JNJ     -0.04%  
  SHW     -0.04%  
  IBM     -0.04%  
  JPM     -0.04%  
  BA      -0.04%  
  GS      -0.04%  
